# IOAI — 2026 Local Onia Churn (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-churn.zip', '/tmp/churn.zip')
    with zipfile.ZipFile('/tmp/churn.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, auc

In [ ]:
seed = 42

root_path = "data"


# Data

In [ ]:
train_df = pd.read_csv(f"{root_path}/train.csv")
test_df = pd.read_csv(f"{root_path}/test.csv")

test_df.head()

# Subtask 1

In [ ]:
# FinancialRiskScore = (Monthly Charge > 70) + (Total Extra Data Charges > 10)

test_df["FinancialRiskScore"] = (test_df["Monthly Charge"] > 70).astype(np.int32) + (
    test_df["Total Extra Data Charges"] > 10
).astype(np.int32)

train_df["FinancialRiskScore"] = (train_df["Monthly Charge"] > 70).astype(np.int32) + (
    train_df["Total Extra Data Charges"] > 10
).astype(np.int32)

subtask1 = test_df["FinancialRiskScore"]

# Subtask 2

In [ ]:
# ServiceQualityScore = (Avg Speed < 50) + (Ping Score > 80) + (Link Quality Index < 30)

test_df["ServiceQualityScore"] = (
    (test_df["Avg Speed"] < 50).astype(np.int32)
    + (test_df["Ping Score"] > 80).astype(np.int32)
    + (test_df["Link Quality Index"] < 30).astype(np.int32)
)

train_df["ServiceQualityScore"] = (
    (train_df["Avg Speed"] < 50).astype(np.int32)
    + (train_df["Ping Score"] > 80).astype(np.int32)
    + (train_df["Link Quality Index"] < 30).astype(np.int32)
)

subtask2 = test_df["ServiceQualityScore"]

In [ ]:
subtask2.describe()

# Subtask 3

In [ ]:
train_df.select_dtypes(include='object').nunique().sort_values(ascending=False)

In [ ]:
def prep_df(df: pd.DataFrame):
    if "Churn" in df:
        y = df["Churn"]
    else:
        y = None

    df = df.drop(["SampleID", "Customer ID", "City", "Churn"], axis=1, errors='ignore')

    df["lat"] = df["Lat Long"].str.split(", ").apply(lambda x: np.float32(x[0]))
    df["long"] = df["Lat Long"].str.split(", ").apply(lambda x: np.float32(x[1]))

    # 1. impute missiong
    missing_cols = df.columns[df.isna().sum() > 0]
    for col in missing_cols:
        val = df[col].mode() if df[col].dtype == 'object' else df[col].mean()
        df[col] = df[col].fillna(value=val)

    # 2. dummy encoding
    cat_cols = df.select_dtypes(include='object')
    for col in cat_cols:
        if df[col].nunique() > 5:
            df = df.drop([col], axis=1)
            continue
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop([col], axis=1)

    if y is not None:
        return df, y
    return df

In [ ]:
X_train, y_train = prep_df(train_df)

test_ids = test_df["SampleID"]
test_df = prep_df(test_df)

In [ ]:
X_train.isna().sum().sum()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)

In [ ]:
def evaluate(clf):
    cv = cross_val_score(clf, X_train, y_train, scoring='roc_auc', cv=3, n_jobs=-1)
    return cv.mean() - cv.std()

In [ ]:
rf = RandomForestClassifier(n_estimators=400, random_state=seed)
evaluate(rf)

In [ ]:
model = rf
model.fit(X_train, y_train)

subtask3 = rf.predict_proba(test_df)[:, 1]

# Submission

In [ ]:
def build_subtask(sid, ans):
    return pd.DataFrame({"id": test_ids, "subtaskID": sid, "answer": ans})

subtasks = [
    (1, subtask1.astype(np.int32)),
    (2, subtask2.astype(np.int32)),
    (3, subtask3),
]

submission = pd.concat([build_subtask(sid, ans) for sid, ans in subtasks])

In [ ]:
submission.head()

In [ ]:
submission.to_csv(f"{root_path}/submission.csv", index=False)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)